In [18]:
import pandas as pd
import numpy as np

df = pd.read_csv('books.csv')
print(df.head(5))

# 4 lines in the original file had unfortunate formatting (commas included in the author line
# so I removed those commas before loading the file)

   bookID                                              title  \
0       1  Harry Potter and the Half-Blood Prince (Harry ...   
1       2  Harry Potter and the Order of the Phoenix (Har...   
2       4  Harry Potter and the Chamber of Secrets (Harry...   
3       5  Harry Potter and the Prisoner of Azkaban (Harr...   
4       8  Harry Potter Boxed Set  Books 1-5 (Harry Potte...   

                      authors  average_rating        isbn         isbn13  \
0  J.K. Rowling/Mary GrandPré            4.57  0439785960  9780439785969   
1  J.K. Rowling/Mary GrandPré            4.49  0439358078  9780439358071   
2                J.K. Rowling            4.42  0439554896  9780439554893   
3  J.K. Rowling/Mary GrandPré            4.56  043965548X  9780439655484   
4  J.K. Rowling/Mary GrandPré            4.78  0439682584  9780439682589   

  language_code    num_pages  ratings_count  text_reviews_count  \
0           eng          652        2095690               27591   
1           eng         

In [19]:
# let's find the mean rating for all the book ratings in the dataset

mean_rating = df['average_rating'].astype(float).mean()
print(mean_rating)

min_ratings = 1000 # I'm arbitrarily setting a minimum vote count of 1000 so we can use the IMDB formula for popularity

df['weighted_rating'] = ((df['ratings_count']/(df['ratings_count']+min_ratings))*df['average_rating']) + (min_ratings/(df['ratings_count']+min_ratings))*mean_rating

3.9336308079446387


In [20]:
most_popular_books = df.sort_values(by='weighted_rating', ascending=False).head(5)
print(most_popular_books[['bookID', 'title', 'authors', 'average_rating', 'weighted_rating']])

      bookID                                              title  \
6590   24812                     The Complete Calvin and Hobbes   
4          8  Harry Potter Boxed Set  Books 1-5 (Harry Potte...   
6592   24814      It's a Magical World (Calvin and Hobbes  #11)   
6         10       Harry Potter Collection (Harry Potter  #1-6)   
6593   24816  Homicidal Psycho Jungle Cat (Calvin and Hobbes...   

                         authors  average_rating  weighted_rating  
6590              Bill Watterson            4.82         4.793313  
4     J.K. Rowling/Mary GrandPré            4.78         4.760052  
6592              Bill Watterson            4.76         4.726779  
6                   J.K. Rowling            4.73         4.702766  
6593              Bill Watterson            4.72         4.671948  


In [23]:
# For this function, I'll assume we have selected a book by ID for author-based recommendation.

def author_recommendation(df, book_id):
    from sklearn.feature_extraction.text import TfidfVectorizer
    tfidf_matrix = TfidfVectorizer().fit_transform(df['authors'])
    from sklearn.metrics.pairwise import linear_kernel
    distance_matrix = linear_kernel(tfidf_matrix)
    index_val = df.loc[df['bookID'] == book_id].index
    distances = distance_matrix[index_val].ravel()
    recommends = np.argsort(distances)[-11:-1]
    return df.iloc[recommends].sort_values(by='weighted_rating', ascending=False)

# testing with book 24812 which is 'The Complete Calvin and Hobbes' by Bill Watterson
recs = author_recommendation(df, 24812)

print(recs[['title', 'authors', 'weighted_rating']])


                                                   title         authors  \
6590                      The Complete Calvin and Hobbes  Bill Watterson   
6593   Homicidal Psycho Jungle Cat (Calvin and Hobbes...  Bill Watterson   
6594                            The Days Are Just Packed  Bill Watterson   
6591        The Calvin and Hobbes Tenth Anniversary Book  Bill Watterson   
6596   Calvin and Hobbes: Sunday Pages 1985-1995: An ...  Bill Watterson   
6595      The Calvin And Hobbes:  Tenth Anniversary Book  Bill Watterson   
1161                A Short History of Nearly Everything     Bill Bryson   
20     The Mother Tongue: English and How It Got That...     Bill Bryson   
10528                                Journeys in English     Bill Bryson   
5555                         Bill Bryson's African Diary     Bill Bryson   

       weighted_rating  
6590          4.793313  
6593          4.671948  
6594          4.654503  
6591          4.616107  
6596          4.541700  
6595         